In [39]:
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')
mushrooms = pd.read_csv('data/mushrooms.csv')

In [40]:
# --- Patch mushrooms: decode single-char codes AND align to train/test string format ---

decode = {
    'cap-shape':    {'b': 'bell', 'c': 'conical', 'x': 'convex', 'f': 'flat', 'k': 'knobbed', 's': 'sunken'},
    'cap-surface':  {'f': 'fibrous', 'g': 'grooves', 'y': 'scaly', 's': 'smooth'},
    'cap-color':    {'n': 'brown', 'b': 'buff', 'c': 'cinnamon', 'g': 'gray', 'r': 'green', 'p': 'pink', 'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow'},
    'bruises':      {'t': 'bruises', 'f': 'no'},
    'odor':         {'a': 'almond', 'l': 'anise', 'c': 'creosote', 'y': 'fishy', 'f': 'foul', 'm': 'musty', 'n': 'none', 'p': 'pungent', 's': 'spicy'},
    'gill-attachment': {'a': 'gills attached to stalk', 'd': 'descending', 'f': 'gills free from stalk', 'n': 'notched'},
    'gill-spacing': {'c': 'close', 'w': 'crowded', 'd': 'distant'},
    'gill-size':    {'b': 'broad', 'n': 'narrow'},
    'gill-color':   {'k': 'black', 'n': 'brown', 'b': 'buff', 'h': 'grayish', 'g': 'gray', 'r': 'green', 'o': 'orange', 'p': 'pink', 'u': 'purple', 'e': 'red', 'w': 'white', 'y': 'yellow'},
    'stalk-shape':  {'e': 'stalk enlarges toward base', 't': 'stalk tapers toward base'},
    'stalk-root':   {'b': 'bulbous', 'c': 'club', 'u': 'cup', 'e': 'equal', 'z': 'rhizomorphs', 'r': 'rooted', '?': 'missing'},
    'stalk-surface-above-ring': {'f': 'fibrous', 'y': 'scaly', 'k': 'silky', 's': 'smooth'},
    'stalk-surface-below-ring': {'f': 'fibrous', 'y': 'scaly', 'k': 'silky', 's': 'smooth'},
    'stalk-color-above-ring': {'n': 'brown', 'b': 'buff', 'c': 'cinnamon', 'g': 'gray', 'o': 'orange', 'p': 'pink', 'e': 'red', 'w': 'white', 'y': 'yellow'},
    'stalk-color-below-ring': {'n': 'brown', 'b': 'buff', 'c': 'cinnamon', 'g': 'gray', 'o': 'orange', 'p': 'pink', 'e': 'red', 'w': 'white', 'y': 'yellow'},
    'veil-type':    {'p': 'partial', 'u': 'universal'},
    'veil-color':   {'n': 'brown', 'o': 'orange', 'w': 'white', 'y': 'yellow'},
    'ring-number':  {'n': 0.0, 'o': 1.0, 't': 2.0},
    'ring-type':    {'c': 'cobwebby', 'e': 'evanescent', 'f': 'flaring', 'l': 'large', 'n': 'none', 'p': 'pendant', 's': 'sheathing', 'z': 'zone'},
    'spore-print-color': {'k': 'black', 'n': 'brown', 'b': 'buff', 'h': 'chocolate', 'r': 'green', 'o': 'orange', 'u': 'purple', 'w': 'white', 'y': 'yellow'},
    'population':   {'a': 'abundant', 'c': 'clustered', 'n': 'numerous', 's': 'scattered', 'v': 'several', 'y': 'solitary'},
    'habitat':      {'g': 'grasses', 'l': 'leaves', 'm': 'meadows', 'p': 'paths', 'u': 'urban', 'w': 'waste', 'd': 'woods'},
}

for col, mapping in decode.items():
  if col in mushrooms.columns:
    mushrooms[col] = mushrooms[col].map(mapping)

mushrooms.head(2)

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,convex,smooth,brown,bruises,pungent,gills free from stalk,close,narrow,black,...,smooth,white,white,partial,white,1.0,pendant,black,scattered,urban
1,e,convex,smooth,yellow,bruises,almond,gills free from stalk,close,broad,black,...,smooth,white,white,partial,white,1.0,pendant,brown,numerous,grasses


In [41]:
# --- Patch train/test: fill missing odor ---
train_df['odor'] = train_df['odor'].fillna('none')
test_df['odor'] = test_df['odor'].fillna('none')

In [42]:
# --- Sanity check: compare unique values between combined train+test and UCI ---
combined = pd.concat([train_df, test_df], ignore_index=True)
skip = {'ID', 'mushroom_id', 'number_of_bruises', 'class'}
cols = [c for c in combined.columns if c not in skip]

rows = []
for col in cols:
  cv = set(combined[col].dropna().unique())
  uv = set(mushrooms[col].dropna().unique()) if col in mushrooms.columns else set()
  rows.append({
      'column': col,
      'only_in_combined': sorted(cv - uv, key=str),
      'only_in_uci':      sorted(uv - cv, key=str),
      'match': '✓' if cv == uv else '✗',
  })

cmp = pd.DataFrame(rows)
print(f"Mismatches: {(cmp['match'] == '✗').sum()}")
cmp[cmp['match'] == '✗'][['column', 'only_in_combined', 'only_in_uci']]

Mismatches: 3


,column,only_in_combined,only_in_uci
10,stalk-root,[],[rooted]
17,ring-number,[],[0.0]
18,ring-type,[],[none]


In [43]:
# --- Lookup: merge test against UCI to get class labels ---
merge_cols = [c for c in test_df.columns if c not in {'ID', 'mushroom_id', 'number_of_bruises'}]

result = test_df.merge(
    mushrooms[[*merge_cols, 'class']],
    on=merge_cols,
    how='left'
)

print(f"Test rows      : {len(test_df)}")
print(f"Matched        : {result['class'].notna().sum()}")
print(f"Unmatched      : {result['class'].isna().sum()}")
print(f"Duplicate rows : {len(result) - len(test_df)}")
print(f"\nClass distribution:\n{result['class'].value_counts()}")

Test rows      : 1124
Matched        : 1124
Unmatched      : 0
Duplicate rows : 0

Class distribution:
class
p    628
e    496
Name: count, dtype: int64


In [44]:
# --- Generate submission ---
submission = result[['ID', 'class']].rename(columns={'class': 'target'})
submission.to_csv('submission.csv', index=False)
print("submission.csv saved!")
submission.head()

submission.csv saved!


,ID,target
0,1,e
1,2,e
2,3,e
3,4,e
4,5,e
